### Rag Pipelines -> Data Ingestion -> Vectore DB Pipeline

In [7]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [19]:
## read all the pdf's inside the directory

def process_all_pdfs(pdf_directory):
    """"Processes all PDF files in a Directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # find all the pdf files recursively in the directory
    pdf_files=list(pdf_dir.glob('**/*.pdf'))
    
    print(f"Found {len(pdf_files)} PDF Files to Process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader=PyPDFLoader(str(pdf_file))
            documents=loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"]=pdf_file.name
                doc.metadata["file_type"]="pdf"
            
            all_documents.extend(documents)
            print(f"✅ Loaded {len(documents)} Pages")

        except Exception as e:
            print(f"❌ Error Processing {pdf_file.name}: {e}")
            
    print(f"\nFinished Processing. Total Documents: {len(all_documents)}")
    return all_documents
    
# Proces all pdfs in the data Directory
all_pdf_documents=process_all_pdfs("../data")
            

Found 7 PDF Files to Process

Processing: Embeddings.pdf
✅ Loaded 1 Pages

Processing: Loaders.pdf
✅ Loaded 1 Pages

Processing: Nastik_-_Kushal_Mehra.pdf
✅ Loaded 238 Pages

Processing: Parsing.pdf
✅ Loaded 1 Pages

Processing: RAG_Learning_Notes.pdf
✅ Loaded 2 Pages

Processing: The_psychology_of_money_-_Morgan_housel.pdf
✅ Loaded 197 Pages

Processing: Vector_Databases.pdf
✅ Loaded 1 Pages

Finished Processing. Total Documents: 441


In [20]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-06-06T12:35:18+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-06-06T12:35:18+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\Embeddings.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Embeddings.pdf', 'file_type': 'pdf'}, page_content='Embeddings\nEmbeddings convert text into vectors representing meaning.\nModels\nOpenAI, BGE, E5, Sentence Transformers.\nUsage\nUsed for similarity search and retrieval.'),
 Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-06-06T12:35:18+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-06-06T12:35:18+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\Loaders.pdf', 'total_pages': 1, 'page': 0, '

### Chunking


In [21]:
## test splitting get into chunks
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Splits documents into smaller chunks for better RAG performance"""
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n","\n",".","!","?"," ", ""]
        
    )
    
    split_docs=text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # show example of split document
    if split_docs:
        print(f"\nExample Chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
        
    return split_docs
    

In [22]:
chunks=split_documents(all_pdf_documents)

Split 441 documents into 1170 chunks

Example Chunk:
Content: Embeddings
Embeddings convert text into vectors representing meaning.
Models
OpenAI, BGE, E5, Sentence Transformers.
Usage
Used for similarity search and retrieval....
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-06-06T12:35:18+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-06-06T12:35:18+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\Embeddings.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Embeddings.pdf', 'file_type': 'pdf'}


### Embedding & Vector DB Insertion
### text->Number Vector

In [24]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [31]:
class EmbeddingManager:
    """Handles document embedding generation using Sentence Transformers"""
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        
        """
        self.model_name=model_name
        self.model=None
        self._load_model()
        
    def _load_model(self):
        """ Load the Sentence Transformer Model """
        try:
            print(f"Loading Embedding Model:{self.model_name}...")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model Loaded Succesfully. Embedding Dimension :{self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error Loading Model {self.model_name}: {e}")
            raise
    
    def generate_embeddings(self,texts:List[str])->np.ndarray:
        """ Generate embeddings for a list of texts 
        Args:
             texts: List of strings to embed
             
        Returns: 
            Numpy array of embeddings with shaape (len(texts), embedding_dimension)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings=self.model.encode(texts, show_progress_bar=True)
        print(f"Generated Embeddings with shape: {embeddings.shape}")
        return embeddings
    
## initialize the embedding manager
embedding_manager=EmbeddingManager()
embedding_manager
    
    

Loading Embedding Model:all-MiniLM-L6-v2...


c:\Users\vikas\OneDrive\Desktop\Ai Agentic Code\RAG_Implementation\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vikas\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6

Model Loaded Succesfully. Embedding Dimension :384


C:\Users\vikas\AppData\Local\Temp\ipykernel_22700\1110271288.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model Loaded Succesfully. Embedding Dimension :{self.model.get_sentence_embedding_dimension()}")


## Vector Store